# 4Field Line Cross Scattering

This notebook tests auto, cross, and total scattering for two line networks `L_12` and `L_34` built from four correlated Gaussian random-wave fields. Wave statistics, radial `k` sampling, and field-label correlations are tunable below.


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import rw_line_scattering as ris

plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
})


## Parameters


In [ ]:
k0_nominal = 1.0
k_distribution = "gaussian_radial"
num_modes_k = 2**10
r_sigma_k = 0.1
seed = 12345

# Options: "qmc", "random", or "quadrature".
k_sampling = "qmc"

if k_sampling == "quadrature":
    k_radii, k_weights = ris.make_radial_k_quadrature(
        num_modes_k,
        k_distribution,
        k0=k0_nominal,
        sigma_k=r_sigma_k*k0_nominal,
    )
else:
    k_rng = np.random.default_rng(seed)
    k_sets = ris.make_field_k_sets(
        num_modes_k,
        k_distribution,
        k_rng,
        k0=k0_nominal,
        r_sigma_k=r_sigma_k,
        shared_k_vectors=True,
        use_qmc_k=(k_sampling == "qmc"),
        qmc_seed=seed,
    )
    k_radii = ris.k_radii_from_vectors(k_sets.psi1)
    k_weights = None

a = ris.gradient_variance_from_k_radii(k_radii, k_weights=k_weights)
rho0 = ris.rho0_from_k_radii(k_radii, k_weights=k_weights)
k_eff = np.sqrt(3*a)

k_mean = np.mean(k_radii) if k_weights is None else np.dot(k_weights, k_radii)
k_std = np.std(k_radii) if k_weights is None else np.sqrt(np.dot(k_weights, (k_radii-k_mean)**2))

r_min = 1e-3 / k_eff
r_max = 5e2 / k_eff
Nr = 5000
r_grid = np.linspace(r_min, r_max, Nr)

Q_min = 0.1 * k_eff
Q_max = 20.0 * k_eff
NQ = 256
Q_grid = np.logspace(np.log10(Q_min), np.log10(Q_max), NQ)

tail_start = 0.8 * r_max

# Conditional-Jacobian sampling.
N_samp = 2**15

# Field-label correlations between psi_1/psi_3 and psi_2/psi_4.
rho_pairs = [
    (0.0, 0.0),
    (1.0, 0.0),
    (1.0, 0.5),
]

print(f"distribution = {k_distribution}")
print(f"k sampling = {k_sampling}")
print(f"nominal k0 = {k0_nominal:.12g}")
print(f"effective k_eff = sqrt(<k^2>) = {k_eff:.12g}")
print(f"mean |k| = {k_mean:.12g}")
print(f"std |k| = {k_std:.12g}")
print(f"a = <k^2>/3 = {a:.12g}")
print(f"rho0 = a/pi = {rho0:.12g}")
print(f"rho0^2 = {rho0**2:.12g}")
print(f"r grid: {len(r_grid)} points from {r_grid[0]:.4g} to {r_grid[-1]:.4g}")
print(f"Q/k grid: {Q_grid[0]/k_eff:.4g} to {Q_grid[-1]/k_eff:.4g}")
print(f"QMC samples = {N_samp}")
print(f"rho pairs = {rho_pairs}")


## Covariance


In [ ]:
wave_stats = ris.compute_wave_stats_for_r_grid(r_grid, k_radii, k_weights=k_weights)
g_num = wave_stats["g"]
gp_num = wave_stats["gp"]
gpp_num = wave_stats["gpp"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].hist(k_radii, bins=80, density=True, weights=k_weights, alpha=0.75)
axes[0].axvline(k0_nominal, color="tab:orange", linestyle="--", label="nominal k0")
axes[0].axvline(k_eff, color="tab:green", linestyle="--", label="sqrt(<k^2>)")
axes[0].set_xlabel("|k|")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=8)

axes[1].plot(r_grid*k_eff, g_num, label="g")
axes[1].plot(r_grid*k_eff, gp_num/k_eff, label="g'/k_eff")
axes[1].plot(r_grid*k_eff, gpp_num/k_eff**2, label="g''/k_eff^2")
axes[1].set_xlabel("r k_eff")
axes[1].set_xscale("log")
axes[1].legend(fontsize=8)
plt.show()


## Self And Cross Correlations


In [ ]:
xi12 = ris.make_qmc_normals(12, N_samp, seed)
W_tail = ris.tail_window(r_grid, tail_start, r_max)

self_result = ris.compute_self_correlation_from_wave_stats(
    r_grid,
    a,
    g_num,
    gp_num,
    gpp_num,
    xi12,
    progress=True,
)
C_self = self_result.C
C_self_coherent = C_self - rho0**2
I_AA = ris.hankel_transform(r_grid, C_self_coherent * W_tail, Q_grid)
I_self = I_AA
I_total_self_ref = 2*I_AA

cross_results = []
for rho13, rho24 in rho_pairs:
    result = ris.compute_cross_correlation_from_wave_stats(
        r_grid,
        a,
        g_num,
        gp_num,
        gpp_num,
        rho13,
        rho24,
        xi12,
        progress=True,
    )
    C_coherent = result.C - rho0**2
    I_AB = ris.hankel_transform(r_grid, C_coherent * W_tail, Q_grid)
    I_total = 2*I_AA + 2*I_AB
    cross_results.append({
        "rho13": rho13,
        "rho24": rho24,
        "C": result.C,
        "C_coherent": C_coherent,
        "I_AB": I_AB,
        "I_total": I_total,
        "I": I_AB,
        "M": result.M,
        "pzeros": result.pzeros,
    })

independent = next((item for item in cross_results if item["rho13"] == 0.0 and item["rho24"] == 0.0), None)
if independent is not None:
    print(f"independent max |C_cross-rho0^2| = {np.max(np.abs(independent['C']-rho0**2)):.6g}")


## Real-Space Checks


In [ ]:
fig, ax = plt.subplots()
ax.semilogx(r_grid*k_eff, C_self, linewidth=2.0, label="self")
ax.axhline(rho0**2, color="black", linestyle=":", label="rho0^2")
ax.semilogx(r_grid*k_eff, C_self_coherent, alpha=0.8, label="self-rho0^2")
ax.set_xlabel("r k_eff")
ax.set_ylabel("C(r)")
ax.set_yscale("log")
ax.legend()
plt.show()

fig, ax = plt.subplots()
for item in cross_results:
    label = f"({item['rho13']}, {item['rho24']})"
    ax.semilogx(r_grid*k_eff, item["C"], label=label)
ax.axhline(rho0**2, color="black", linestyle=":", label="rho0^2")
ax.set_xlabel("r k_eff")
ax.set_ylabel("C_cross")
ax.set_yscale("log")
ax.legend(fontsize=8)
plt.show()

fig, ax = plt.subplots()
for item in cross_results:
    label = f"({item['rho13']}, {item['rho24']})"
    ax.semilogx(r_grid*k_eff, item["C_coherent"], label=label)
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_xlabel("r k_eff")
ax.set_ylabel("C_cross-rho0^2")
ax.set_yscale("symlog", linthresh=1e-4)
ax.legend(fontsize=8)
plt.show()


## Scattering


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

axes[0].loglog(Q_grid/k_eff, np.abs(I_AA), color="black", linewidth=2.2, label="$I_{AA}$")
axes[0].set_title("auto")
axes[0].set_xlabel(r"$Q/k$")
axes[0].set_ylabel(r"$I(Q)$")
axes[0].set_ylim(1e-2,1)
axes[0].legend(fontsize=8)

for item in cross_results:
    label = f"({item['rho13']}, {item['rho24']})"
    axes[1].loglog(Q_grid/k_eff, np.abs(item["I_AB"]), label=label)
    axes[2].loglog(Q_grid/k_eff, np.abs(item["I_total"]), label=label)
axes[2].loglog(Q_grid/k_eff, np.abs(I_total_self_ref), color="black", linestyle="--", linewidth=2.0, label="$2I_{AA}$")

axes[1].set_title("cross")
axes[1].set_xlabel(r"$Q/k$")
axes[1].set_ylabel(r"$I_{AB}(Q)$")
axes[1].set_ylim(1e-4,1)
axes[1].legend(fontsize=8)

axes[2].set_title(r"$I_{AA} + I_{BB} + 2 I_{AB}$")
axes[2].set_xlabel(r"$Q/k$")
axes[2].set_ylabel(r"$I_{total}(Q)$")
axes[2].set_ylim(1e-2,1)    
axes[2].legend(fontsize=8)

for ax in axes:
    ax.axvline(1.0, color="black", linestyle=":", linewidth=0.8)
    ax.axvline(2.0, color="black", linestyle="--", linewidth=0.8)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

axes[0].semilogx(Q_grid/k_eff, Q_grid*I_AA, color="black", linewidth=2.2, label="$Q I_{AA}$")
axes[0].axhline(a, color="black", linestyle="--", label="self plateau")
axes[0].set_title("auto")
axes[0].set_xlabel(r"$Q/k$")
axes[0].set_ylabel("Q I(Q)")
axes[0].legend(fontsize=8)

for item in cross_results:
    label = f"({item['rho13']}, {item['rho24']})"
    axes[1].semilogx(Q_grid/k_eff, Q_grid*item["I_AB"], label=label)
    axes[2].semilogx(Q_grid/k_eff, Q_grid*item["I_total"], label=label)
axes[2].semilogx(Q_grid/k_eff, Q_grid*I_total_self_ref, color="black", linestyle="--", linewidth=2.0, label="$2 I_{AA}$")

axes[1].set_title("cross")
axes[1].set_xlabel(r"$Q/k$")
axes[1].set_ylabel("Q I_{AB}(Q)")
axes[1].legend(fontsize=8)

axes[2].set_title(r"$I_{AA} + I_{BB} + 2 I_{AB}$")
axes[2].set_xlabel(r"$Q/k$")
axes[2].set_ylabel("Q I_{total}(Q)")
axes[2].legend(fontsize=8)

for ax in axes:
    ax.axvline(1.0, color="black", linestyle=":", linewidth=0.8)
    ax.axvline(2.0, color="black", linestyle="--", linewidth=0.8)
plt.show()


## Save Outputs


In [ ]:
output_dir = Path("rw_4field_line_demo_output")
output_dir.mkdir(exist_ok=True)

rho_pair_array = np.array([[item["rho13"], item["rho24"]] for item in cross_results], dtype=float)
np.savez_compressed(
    output_dir / "4field_line_demo_data.npz",
    r_grid=r_grid,
    Q_grid=Q_grid,
    k0_nominal=k0_nominal,
    k_distribution=k_distribution,
    k_sampling=k_sampling,
    r_sigma_k=r_sigma_k,
    k_radii=k_radii,
    k_weights=k_weights,
    a=a,
    k_eff=k_eff,
    rho0=rho0,
    rho_pairs=rho_pair_array,
    g=g_num,
    gp=gp_num,
    gpp=gpp_num,
    C_self=C_self,
    C_self_coherent=C_self_coherent,
    I_AA=I_AA,
    I_self=I_self,
    I_total_self_ref=I_total_self_ref,
    M_self=self_result.M,
    pzeros_self=self_result.pzeros,
    C_cross=np.array([item["C"] for item in cross_results]),
    C_cross_coherent=np.array([item["C_coherent"] for item in cross_results]),
    I_AB=np.array([item["I_AB"] for item in cross_results]),
    I_cross=np.array([item["I_AB"] for item in cross_results]),
    I_total=np.array([item["I_total"] for item in cross_results]),
    M_cross=np.array([item["M"] for item in cross_results]),
    pzeros_cross=np.array([item["pzeros"] for item in cross_results]),
    tail_start=tail_start,
    N_samp=N_samp,
    seed=seed,
)
print(f"saved data to {output_dir.resolve()}")
